# 03 - Ingeniería de Atributos (Feature Engineering)

En este notebook, exploraremos la creación de nuevas características derivadas de los datos originales. La ingeniería de atributos es uno de los pasos más importantes en el aprendizaje automático, ya que a menudo permite que los modelos capturen relaciones más complejas y mejoren significativamente su precisión.

## 1. Configuración e Importación de Bibliotecas
Comenzamos importando las herramientas necesarias para el análisis de datos y la visualización.

In [1]:
# Importación de bibliotecas estándar para análisis de datos
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Ajuste de parámetros globales para que las gráficas sean legibles y estéticas
plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

## 2. Carga del Conjunto de Datos
Cargamos los datos que dividimos previamente en el notebook anterior para trabajar sobre el set de entrenamiento.

In [2]:
# Cargamos la ruta del directorio de datos procesados (interim)
from housing.config import SPLIT_DIR

# Lectura del archivo CSV correspondiente al set de entrenamiento crudo
df_raw = pd.read_csv(SPLIT_DIR / "housing_train_raw.csv")

# Realizamos una copia profunda para evitar modificar accidentalmente el objeto original
df = df_raw.copy()

## 3. Creación de Nuevas Características
A menudo, las proporciones o ratios entre variables son más informativos que los valores absolutos. Aquí creamos tres nuevos atributos:
*   **Habitaciones por hogar**: Tamaño promedio de las viviendas.
*   **Ratio de dormitorios**: Proporción de dormitorios respecto al total de habitaciones.
*   **Personas por hogar**: Densidad de ocupación promedio.

In [3]:
# Cálculo de nuevas columnas (Feature Engineering)

# Número promedio de habitaciones por vivienda
df["rooms_per_house"] = df["total_rooms"] / df["households"]

# Proporción de dormitorios respecto al total de habitaciones (indicador de densidad)
df["bedrooms_ratio"] = df["total_bedrooms"] / df["total_rooms"]

# Número promedio de personas por vivienda
df["people_per_house"] = df["population"] / df["households"]

## 4. Análisis de Correlación
Verificamos si estas nuevas variables tienen una correlación fuerte con el valor medio de la vivienda (`median_house_value`).

In [4]:
# Analizamos la matriz de correlación numérica
corr_matrix = df.corr(numeric_only=True)

# Ordenamos las correlaciones con respecto al valor objetivo
corr_matrix["median_house_value"].sort_values(ascending=False)

## 5. Evaluación del Impacto: ¿Ayudó la creación de nuevas variables?

La respuesta es un rotundo **SÍ**. 

Si analizamos el coeficiente de correlación de Pearson con `median_house_value`:

*   **Ratio de dormitorios (`bedrooms_ratio`)**: Obtuvo un coeficiente de **-0.25**. 
    *   *Comparación*: Es mucho más informativo que sus componentes originales: `total_rooms` (0.13) y `total_bedrooms` (0.05).
    *   *Significado*: Esta nueva variable es ahora la **segunda más correlacionada** con el precio de la vivienda (después de los ingresos medios). Indica que cuanto menor es la proporción de dormitorios (casas menos densas o con más áreas sociales), mayor es el valor de la propiedad.
*   **Habitaciones por hogar (`rooms_per_house`)**: Con **0.14**, es un mejor indicador que el conteo total de habitaciones (0.13).

### Conclusión Profesional
La implementación de estas variables fue **altamente beneficiosa**. Hemos logrado transformar datos de conteo simple, que por sí solos no aportaban mucho valor, en **indicadores potentes** que el modelo de aprendizaje automático podrá aprovechar mucho mejor para realizar predicciones precisas.

## 6. Visualización Final
Confirmamos visualmente la fuerza de estas nuevas relaciones mediante un mapa de calor.

In [5]:
# Selección de atributos clave para la visualización gráfica
attributes_feauture = ["median_house_value", "median_income", "rooms_per_house",
 "housing_median_age", "bedrooms_ratio", "people_per_house"]

import seaborn as sns

# Creación del mapa de calor
corr = df[attributes_feauture].corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.title("Mapa de Calor: Correlaciones con Nuevos Atributos")
plt.tight_layout()
plt.show()